# CNN Architectures Evolution: LeNet → AlexNet → VGG → Inception

## Introduction

This notebook traces the **historical evolution of CNN architectures** that transformed computer vision from a challenging research problem to production-ready systems.

**What we'll learn:**
- **LeNet (1998)**: The pioneering CNN that proved convolutions work for digit recognition
- **AlexNet (2012)**: The ImageNet breakthrough that showed deep CNNs + ReLU + Dropout scale
- **VGG (2014)**: Depth matters - stacking small 3×3 filters beats large filters
- **Inception (2014)**: Multi-scale feature extraction with parallel convolutions

**Why this matters:**

Each architecture introduced innovations that became **standard practice**. Understanding the progression shows *why* modern CNNs are designed the way they are.

We'll implement each architecture and train them on CIFAR-10 to compare their effectiveness.

## 1. Setup

### Import libraries

We'll use PyTorch for implementations and the shared library for dataset utilities.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from aiml_notebooks import get_device, set_seed

### Set random seed for reproducibility

In [ ]:
set_seed(42)

### Configure device

Use GPU if available for faster training.

In [ ]:
device = get_device()
print(f"Using device: {device}")

## 2. Dataset Preparation

We'll use **CIFAR-10** (32×32 color images, 10 classes) as our benchmark dataset. All architectures will be adapted to work with this small image size.

**Note:** Original papers used different datasets (LeNet: MNIST, AlexNet: ImageNet), but we standardize on CIFAR-10 for fair comparison.

### Define data transforms

Normalize images to have zero mean and unit variance for stable training.

In [ ]:
# CIFAR-10 normalization statistics
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

### Load CIFAR-10 dataset

In [ ]:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

### Create data loaders

In [ ]:
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

### Visualize sample images

Let's see what CIFAR-10 images look like.

In [ ]:
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Get a batch and denormalize for visualization
images, labels = next(iter(train_loader))

def denormalize(img):
    img = img * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)
    return img.clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for idx, ax in enumerate(axes.flat):
    img = denormalize(images[idx])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(classes[labels[idx]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Architecture 1: LeNet (1998)

### The Pioneer

**LeNet-5** (Yann LeCun, 1998) was the first successful CNN, designed for handwritten digit recognition. It proved that:
1. **Convolutional layers** can learn hierarchical features automatically
2. **Pooling** reduces spatial dimensions while preserving important features
3. End-to-end gradient descent training works for deep(ish) networks

**Original architecture:**
- Input: 32×32 grayscale
- Conv(6) → Pool → Conv(16) → Pool → FC(120) → FC(84) → FC(10)
- Used tanh activation

**Our adaptation:**
- Modified for 32×32 RGB images (CIFAR-10)
- Use ReLU instead of tanh (modern practice)
- Batch normalization for training stability

### Implement LeNet

A simple stack: Conv → Pool → Conv → Pool → Flatten → FC layers.

In [ ]:
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Feature extraction: convolutional layers
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(6)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Classification: fully connected layers
        self.fc1 = nn.Linear(16 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
    
    def forward(self, x):
        # Feature extraction
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # 32x32x3 -> 16x16x6
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # 16x16x6 -> 6x6x16
        
        # Flatten
        x = x.view(x.size(0), -1)  # Flatten to [batch, 576]
        
        # Classification
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # No softmax - included in CrossEntropyLoss
        return x

### Count parameters

LeNet is tiny by modern standards - only ~60K parameters!

In [ ]:
model = LeNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"LeNet parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

## 4. Architecture 2: AlexNet (2012)

### The ImageNet Breakthrough

**AlexNet** (Krizhevsky et al., 2012) won ImageNet 2012 by a huge margin, proving deep CNNs could scale. Key innovations:

1. **ReLU activation**: Trains 6× faster than tanh, enables deeper networks
2. **Dropout**: Randomly drops neurons during training to prevent overfitting
3. **Data augmentation**: Random crops, flips, color jittering
4. **GPU training**: Used 2 GTX 580 GPUs (split model across GPUs)
5. **Local Response Normalization** (LRN): Lateral inhibition between feature maps (less common now)

**Original architecture:**
- Input: 224×224 RGB
- 5 conv layers, 3 FC layers
- 60M parameters

**Our adaptation:**
- Scaled down for 32×32 CIFAR-10 images
- Removed LRN (replaced by BatchNorm)
- Smaller filters to match image size

### Implement AlexNet

Deeper than LeNet with 5 conv layers, ReLU, and Dropout.

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Feature extraction: 5 convolutional layers
        self.features = nn.Sequential(
            # Conv1: 32x32x3 -> 32x32x64
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 32x32 -> 16x16
            
            # Conv2: 16x16x64 -> 16x16x192
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 16x16 -> 8x8
            
            # Conv3: 8x8x192 -> 8x8x384
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),
            
            # Conv4: 8x8x384 -> 8x8x256
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Conv5: 8x8x256 -> 8x8x256
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 8x8 -> 4x4
        )
        
        # Classification: 3 fully connected layers with dropout
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

### Count parameters

AlexNet is much larger - around 10M parameters even in our scaled-down version.

In [ ]:
model = AlexNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"AlexNet parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"\nIncrease vs LeNet: {total_params / 61706:.1f}x")

## 5. Architecture 3: VGG (2014)

### Depth with Small Filters

**VGG** (Simonyan & Zisserman, 2014) showed that **depth matters** more than filter size. Key insight:

**Two 3×3 convs have the same receptive field as one 5×5 conv, but:**
- Fewer parameters (2×(3×3) = 18 vs 5×5 = 25)
- More non-linearity (2 ReLU layers vs 1)
- Better feature learning

**VGG variants:**
- VGG-11: 11 layers (8 conv + 3 FC)
- VGG-13: 13 layers (10 conv + 3 FC)
- VGG-16: 16 layers (13 conv + 3 FC) ← Most popular
- VGG-19: 19 layers (16 conv + 3 FC)

**Design principles:**
1. All conv filters are 3×3 with stride 1, padding 1
2. MaxPool 2×2 with stride 2 halves spatial dimensions
3. Channels double after each pooling: 64 → 128 → 256 → 512 → 512
4. Very simple, uniform architecture

**Our implementation:**
- VGG-11 adapted for CIFAR-10 (32×32 images)
- BatchNorm instead of original architecture (VGG didn't use it)
- Smaller FC layers to reduce parameters

### Implement VGG-11

Stacks of 3×3 convs separated by pooling. Very regular structure.

In [ ]:
class VGG11(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # VGG-11 configuration: [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']
        # 'M' means MaxPool
        
        self.features = nn.Sequential(
            # Block 1: 32x32 -> 16x16
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 8x8 -> 4x4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 4: 4x4 -> 2x2
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 5: 2x2 -> 1x1
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512 * 1 * 1, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

### Count parameters

VGG is much deeper but still manageable for CIFAR-10.

In [ ]:
model = VGG11().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"VGG-11 parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

## 6. Architecture 4: Inception (2014)

### Multi-Scale Feature Extraction

**Inception (GoogLeNet)** (Szegedy et al., 2014) asked: "What filter size should we use?" Answer: **Use all of them!**

**Key innovation: Inception module**

Run multiple convolution sizes **in parallel** and concatenate:
- 1×1 conv (capture point-wise features)
- 3×3 conv (capture local patterns)
- 5×5 conv (capture larger patterns)
- 3×3 MaxPool (preserve spatial features)

**Problem:** This is computationally expensive!

**Solution:** Use **1×1 convolutions** as "bottleneck layers" to reduce channels before expensive operations.

**Benefits:**
1. Multi-scale feature extraction
2. Fewer parameters than VGG despite being deeper (uses 1×1 convs)
3. No fully-connected layers (uses Global Average Pooling instead)
4. More efficient: 22 layers with only 5M parameters

**Our implementation:**
- Simplified Inception module
- Adapted for CIFAR-10's small 32×32 images
- 2 inception modules to show the concept

### Implement Inception Module

The core building block: parallel convolutions with different receptive fields.

In [ ]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool):
        """Inception module with dimension reduction.
        
        Args:
            in_channels: Input channels
            out_1x1: Output channels for 1x1 conv branch
            red_3x3: Reduction channels before 3x3 conv
            out_3x3: Output channels for 3x3 conv branch
            red_5x5: Reduction channels before 5x5 conv
            out_5x5: Output channels for 5x5 conv branch
            out_pool: Output channels for pool projection branch
        """
        super().__init__()
        
        # 1x1 conv branch
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.BatchNorm2d(out_1x1),
            nn.ReLU(inplace=True),
        )
        
        # 3x3 conv branch with 1x1 reduction
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, red_3x3, kernel_size=1),
            nn.BatchNorm2d(red_3x3),
            nn.ReLU(inplace=True),
            nn.Conv2d(red_3x3, out_3x3, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_3x3),
            nn.ReLU(inplace=True),
        )
        
        # 5x5 conv branch with 1x1 reduction
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, red_5x5, kernel_size=1),
            nn.BatchNorm2d(red_5x5),
            nn.ReLU(inplace=True),
            nn.Conv2d(red_5x5, out_5x5, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_5x5),
            nn.ReLU(inplace=True),
        )
        
        # MaxPool branch with 1x1 projection
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1),
            nn.BatchNorm2d(out_pool),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        # Run all branches in parallel
        branch1 = self.branch1(x)
        branch2 = self.branch2(x)
        branch3 = self.branch3(x)
        branch4 = self.branch4(x)
        
        # Concatenate along channel dimension
        outputs = [branch1, branch2, branch3, branch4]
        return torch.cat(outputs, 1)

### Implement Inception Network

Stack inception modules with occasional pooling for downsampling.

In [ ]:
class InceptionNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Initial conv layers
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.maxpool1 = nn.MaxPool2d(2, 2)  # 32x32 -> 16x16
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
        )
        self.maxpool2 = nn.MaxPool2d(2, 2)  # 16x16 -> 8x8
        
        # Inception modules
        # inception3a: 192 -> 256 channels
        self.inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32)
        # inception3b: 256 -> 480 channels
        self.inception3b = InceptionModule(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool2d(2, 2)  # 8x8 -> 4x4
        
        # Global Average Pooling instead of FC layers
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(480, num_classes)
    
    def forward(self, x):
        # Initial convolutions
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.conv2(x)
        x = self.maxpool2(x)
        
        # Inception modules
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)
        
        # Global average pooling + classifier
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

### Count parameters

Inception is efficient - fewer parameters than VGG despite multi-scale features.

In [ ]:
model = InceptionNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"InceptionNet parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

## 7. Training Utilities

We'll create reusable training and evaluation functions to compare all architectures fairly.

### Define training function

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({'loss': f'{running_loss/len(pbar):.3f}', 
                         'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / len(train_loader), 100. * correct / total

### Define evaluation function

In [ ]:
def evaluate(model, test_loader, criterion, device):
    """Evaluate on test set."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(test_loader), 100. * correct / total

### Define full training loop

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    """Train a model and return history."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                           factor=0.5, patience=2)
    
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Evaluate
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        
        # Learning rate scheduling
        scheduler.step(test_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
    
    return history

## 8. Training All Architectures

Now let's train all four architectures and compare their performance!

### Train LeNet

In [ ]:
print("=" * 60)
print("Training LeNet")
print("=" * 60)

lenet = LeNet().to(device)
lenet_history = train_model(lenet, train_loader, test_loader, epochs=15, lr=0.001)

### Train AlexNet

In [ ]:
print("\n" + "=" * 60)
print("Training AlexNet")
print("=" * 60)

alexnet = AlexNet().to(device)
alexnet_history = train_model(alexnet, train_loader, test_loader, epochs=15, lr=0.001)

### Train VGG-11

In [ ]:
print("\n" + "=" * 60)
print("Training VGG-11")
print("=" * 60)

vgg = VGG11().to(device)
vgg_history = train_model(vgg, train_loader, test_loader, epochs=15, lr=0.001)

### Train InceptionNet

In [ ]:
print("\n" + "=" * 60)
print("Training InceptionNet")
print("=" * 60)

inception = InceptionNet().to(device)
inception_history = train_model(inception, train_loader, test_loader, epochs=15, lr=0.001)

## 9. Comparison and Analysis

Let's compare all architectures across multiple dimensions.

### Plot training curves

Visualize loss and accuracy over time for all models.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

histories = {
    'LeNet': lenet_history,
    'AlexNet': alexnet_history,
    'VGG-11': vgg_history,
    'Inception': inception_history
}

# Plot train loss
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
for name, hist in histories.items():
    axes[0, 0].plot(hist['train_loss'], marker='o', label=name, linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot test loss
axes[0, 1].set_title('Test Loss', fontsize=14, fontweight='bold')
for name, hist in histories.items():
    axes[0, 1].plot(hist['test_loss'], marker='o', label=name, linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot train accuracy
axes[1, 0].set_title('Training Accuracy', fontsize=14, fontweight='bold')
for name, hist in histories.items():
    axes[1, 0].plot(hist['train_acc'], marker='o', label=name, linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot test accuracy
axes[1, 1].set_title('Test Accuracy', fontsize=14, fontweight='bold')
for name, hist in histories.items():
    axes[1, 1].plot(hist['test_acc'], marker='o', label=name, linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Final performance comparison

Compare final test accuracy and parameter count.

In [ ]:
models = {
    'LeNet': lenet,
    'AlexNet': alexnet,
    'VGG-11': vgg,
    'Inception': inception
}

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print(f"{'Architecture':<15} {'Parameters':<15} {'Test Accuracy':<15} {'Efficiency'}")
print("-"*70)

for name, model in models.items():
    params = sum(p.numel() for p in model.parameters())
    test_acc = histories[name]['test_acc'][-1]
    efficiency = test_acc / (params / 1e6)  # Accuracy per million params
    print(f"{name:<15} {params:>12,}   {test_acc:>12.2f}%   {efficiency:>10.2f}")

print("="*70)

### Visualize architecture comparison

Bar chart showing parameters vs accuracy.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(models.keys())
params = [sum(p.numel() for p in models[name].parameters()) / 1e6 for name in names]
accs = [histories[name]['test_acc'][-1] for name in names]

# Parameters bar chart
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
ax1.bar(names, params, color=colors)
ax1.set_ylabel('Parameters (Millions)')
ax1.set_title('Model Size Comparison', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Accuracy bar chart
ax2.bar(names, accs, color=colors)
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('Model Performance Comparison', fontweight='bold')
ax2.set_ylim([0, 100])
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for ax, values in [(ax1, params), (ax2, accs)]:
    for i, v in enumerate(values):
        ax.text(i, v + max(values)*0.02, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Key Insights

### What We Learned

**1. LeNet (1998): The Foundation**
- Proved CNNs work for vision tasks
- Simple but effective: conv → pool → fc pattern
- Limited by shallow depth and small capacity

**2. AlexNet (2012): The Breakthrough**
- **ReLU** enables deeper networks (no vanishing gradients)
- **Dropout** prevents overfitting in large models
- **Data augmentation** is crucial for generalization
- Showed that scale (depth + data) matters

**3. VGG (2014): Depth Wins**
- **Small filters (3×3) stacked deeply** beat large filters
- More non-linearity (ReLU layers) = better feature learning
- Simple, uniform architecture is easier to design
- But: many parameters in FC layers

**4. Inception (2014): Efficiency Matters**
- **Multi-scale features** capture patterns at different scales
- **1×1 convolutions** reduce computation ("bottleneck" layers)
- **Global Average Pooling** eliminates FC layers
- Better accuracy-per-parameter than VGG

### The Evolution Pattern

Each architecture solved problems from the previous generation:
- **LeNet → AlexNet**: Scale up with ReLU + Dropout
- **AlexNet → VGG**: Go deeper with small filters
- **VGG → Inception**: Be efficient with multi-scale + 1×1 convs

### What Came Next?

The evolution didn't stop here:
- **ResNet (2015)**: Skip connections enable 100+ layer networks
- **DenseNet (2017)**: Dense connections for feature reuse
- **EfficientNet (2019)**: Compound scaling (depth + width + resolution)
- **Vision Transformers (2021)**: Attention replaces convolutions

But all modern architectures build on these foundational ideas!

## 11. Interactive Exploration

Try these experiments to deepen your understanding:

**Experiment 1: Remove innovations**
- Train AlexNet without Dropout → See overfitting increase
- Use sigmoid instead of ReLU → Training gets much slower

**Experiment 2: Architecture modifications**
- Replace VGG's 3×3 filters with 5×5 → More parameters, similar accuracy
- Remove 1×1 convs from Inception → Parameters explode

**Experiment 3: Scale experiments**
- Make VGG wider (more channels) vs deeper (more layers)
- Compare parameter efficiency

**Experiment 4: Different datasets**
- Try these architectures on MNIST (easier) or ImageNet (harder)
- See how relative performance changes

## 12. Summary

This notebook traced the **evolution of CNN architectures** from LeNet (1998) to Inception (2014).

**Key takeaways:**

1. **Activation functions matter**: ReLU unlocked deep networks
2. **Regularization is essential**: Dropout prevents overfitting at scale
3. **Depth beats width**: Stacking small filters works better than large filters
4. **Efficiency matters**: 1×1 convs and global pooling reduce parameters
5. **Multi-scale features**: Parallel convolutions capture different patterns

Each architecture introduced innovations that became **standard practice** in modern deep learning. Understanding this progression helps you:
- Design better architectures
- Debug training issues
- Choose the right model for your task

The journey continues with ResNet, which solved the depth problem with skip connections - covered in the residual-connections notebook!